# Game Theory Simulation: Market Entry Game

## Assignment 2: Intro to Agent Building

This notebook contains my completed rules-based agent, tournament runs, and reflection. The agent uses opponent history to balance cooperation with protection against repeated undercutting.

## Game Setup

Each round has two choices: **INVEST** in developing the market or **UNDERCUT** with aggressive pricing. The payoff matrix is:

| | Competitor Invests | Competitor Undercuts |
|---|---:|---:|
| **I Invest** | (3, 3) | (0, 5) |
| **I Undercut** | (5, 0) | (1, 1) |

The game is a repeated Prisoner's Dilemma. Because future rounds matter, an agent can use past actions to decide whether cooperation is still worth trusting.

In [1]:
import random
from agents import *
from game_engine import Game, Tournament

random.seed(7)
print('Game theory simulation ready!')


## Custom Agent: Adaptive Forgiving

My strategy starts by investing, then watches the opponent's recent behavior. Instead of reacting to one isolated undercut, it looks at the last five rounds. It undercuts when the opponent has been consistently aggressive, but returns to investing when cooperation becomes common again. This is meant to handle noisy decisions better than a strategy that immediately retaliates forever.

In [2]:
class MyCustomAgent(Agent):
    """Adaptive rules-based agent that cooperates but protects itself from sustained defection."""
    def __init__(self, noise=0.0):
        super().__init__('Adaptive Forgiving',
                         'Uses recent opponent behavior with forgiveness for isolated defections',
                         noise)

    def choose_action(self) -> bool:
        # Build trust at the beginning of each match.
        if self.round_num < 3:
            action = INVEST
        else:
            recent = self.history[-5:]
            cooperation_rate = sum(recent) / len(recent)

            # Protect against sustained undercutting.
            if cooperation_rate < 0.40:
                action = UNDERCUT
            # React to moderate but recent aggression.
            elif self.history[-1] == UNDERCUT and cooperation_rate < 0.70:
                action = UNDERCUT
            # Forgive isolated defections and keep cooperation going.
            else:
                action = INVEST

        return self._apply_noise(action)

my_agent = MyCustomAgent()
print(my_agent.name)


In [3]:
# Quick test against a random opponent.
test_game = Game(MyCustomAgent(noise=0.1), RandomAgent(), num_rounds=30)
test_game.play()
print(test_game.score1, test_game.score2)


## Intro Tournament

I compared the custom agent with the pre-built strategies using 100 rounds per match and 10 tournaments. I used a fixed random seed for this notebook so the reported run is reproducible.

In [4]:
# Run the intro tournament with the completed custom agent.
random.seed(7)
agents = [
    AlwaysInvestAgent(),
    AlwaysUndercutAgent(),
    TitForTatAgent(),
    GrimTriggerAgent(),
    PavlovAgent(),
    RandomAgent(0.5),
    TitForTwoTatsAgent(),
    GenerousTitForTatAgent(),
    AdaptiveAgent(),
    MyCustomAgent()
]

tournament = Tournament(agents, rounds_per_match=100, num_tournaments=10)
tournament.run_tournament()
intro_rankings = tournament.get_rankings()
print('\nIntro tournament ranking:')
for i, (name, score) in enumerate(intro_rankings, 1):
    print(f'{i}. {name} - {score}')


Running 10 tournament(s) with 10 agents...

Tournament 1/10
Tournament 2/10
Tournament 3/10
Tournament 4/10
Tournament 5/10
Tournament 6/10
Tournament 7/10
Tournament 8/10
Tournament 9/10
Tournament 10/10
Tournament complete!

Intro tournament ranking:
1. Grim Trigger - 24928
2. Tit-for-Tat - 24249
3. Adaptive Forgiving - 24189
4. Generous Tit-for-Tat - 24126
5. Adaptive - 23960
6. Pavlov - 23825
7. Tit-for-Two-Tats - 23803
8. Always Invest - 22584
9. Random (0.5) - 20273
10. Always Undercut - 18356


## Noisy Tournament

The noisy tournament gives agents a 10% chance of having an intended action flipped. This makes immediate retaliation less reliable because an apparent defection may not reflect the opponent's actual strategy.

In [5]:
random.seed(7)
noisy_agents = [
    AlwaysInvestAgent(noise=0.1),
    AlwaysUndercutAgent(noise=0.1),
    TitForTatAgent(noise=0.1),
    GrimTriggerAgent(noise=0.1),
    PavlovAgent(noise=0.1),
    RandomAgent(0.5),
    TitForTwoTatsAgent(noise=0.1),
    GenerousTitForTatAgent(noise=0.1),
    AdaptiveAgent(noise=0.1),
    MyCustomAgent(noise=0.1)
]

noisy_tournament = Tournament(noisy_agents, rounds_per_match=200, num_tournaments=10)
noisy_tournament.run_tournament()
noisy_rankings = noisy_tournament.get_rankings()
print('\nNoisy tournament ranking:')
for i, (name, score) in enumerate(noisy_rankings, 1):
    print(f'{i}. {name} - {score}')


Running 10 tournament(s) with 10 agents...

Tournament 1/10
Tournament 2/10
Tournament 3/10
Tournament 4/10
Tournament 5/10
Tournament 6/10
Tournament 7/10
Tournament 8/10
Tournament 9/10
Tournament 10/10
Tournament complete!

Noisy tournament ranking:
1. Adaptive Forgiving - 41556
2. Tit-for-Tat - 41376
3. Generous Tit-for-Tat - 40958
4. Adaptive - 40835
5. Pavlov - 40513
6. Tit-for-Two-Tats - 40499
7. Grim Trigger - 40226
8. Always Undercut - 40109
9. Random (0.5) - 40036
10. Always Invest - 36374


## Reflection

### Strategic Analysis

**1. Which strategy performed best in the tournament? Why do you think this is?**

In the intro tournament, Grim Trigger had the highest total score in my run. In the noisy tournament, my Adaptive Forgiving agent had the highest score because it did not overreact to every single flipped action and could still protect itself when undercutting became consistent.

**2. How did your custom agent perform (IE. What was its rank in both tournaments)? What did you change about its strategy to improve its ranking?**

My custom agent placed 3rd in the intro tournament and 1st in the noisy tournament. I used the opponent's recent five-round cooperation rate and only switched to undercutting after sustained aggression, which made the strategy less sensitive to one random bad move.

**3. Did your agent's rankings change significantly in the noisy tournament? How about other agents? Why might that be?**

My agent moved from 3rd to 1st when noise was added, while some other strategies changed more noticeably. Noise makes strategies that depend on exact last-round behavior less reliable because a flipped action can cause the agent to misunderstand what the opponent intended.

**4. Why do you think agents with the highest win percentage didn't place the best in the tournament? Think about what it takes to "Win", and whether that is actually always the optimal strategy.**

A win percentage only counts how often an agent scores more than its opponent, not how many points it earns across all of its matches. An agent can win many close games but lose enough points in other matches that its overall tournament score is lower than an agent that consistently earns strong scores.

### Real-World Applications

**5. Can you think of real business scenarios where these strategies might apply?**

These strategies could apply to companies deciding whether to invest in advertising, infrastructure, or customer education while competitors are making similar decisions. They could also apply to pricing competition, where a company has to decide whether to cooperate with a stable market or respond aggressively to a competitor's price cuts.

**6. The agent you built this unit is a Rules-Based (Heuristic) agent. How do you think a learning or hybrid agent might improve on this heuristic baseline?**

A learning agent could use past tournament data to discover patterns that I did not hardcode into the rules. A hybrid agent could keep the safety rules for obvious situations while learning when cooperation or undercutting produces better long-term results against different opponents.

### Ethical Considerations

**7. Is the "Always Undercut" strategy ever justified in real business?**

It could make sense in a limited situation where a company needs to respond to a serious competitive threat or protect its market position. However, using aggressive pricing all the time could create a price war, reduce long-term investment, and potentially create legal or ethical concerns depending on the market and behavior involved.

**8. What role should reputation and long-term relationships play in business strategy?**

Reputation can affect whether other companies, suppliers, and customers are willing to work with a business in the future. Long-term relationships can make cooperation more valuable because a short-term gain from undercutting may not be worth damaging trust and future opportunities.

## Final Notes

The main change between the two tournaments was adapting the agent to uncertainty. The strategy keeps cooperation as the default but uses a short history window to recognize sustained defection, which is a simple rules-based approach to handling noisy real-world conditions.